# NDPI Download Notebook

Downloads the NDPI whole-slide images that correspond to the GeoJSON annotations in
`data/dataset_28_04/`. All download logic is delegated to `scripts/download_ndpi.py`;
this notebook only imports from it and provides an interactive status view.

**PANGAEA datasets**
- CHN slides → [PANGAEA 984641](https://doi.pangaea.de/10.1594/PANGAEA.984641)
- LHP slides → [PANGAEA 984640](https://doi.pangaea.de/10.1594/PANGAEA.984640)

**Outputs** saved to `data/ndpi/` (one `.ndpi` per GeoJSON).

In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Make project root importable regardless of where the notebook is launched from
PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
# Import all download logic from the existing script — nothing is reimplemented here.
from scripts.download_ndpi import (
    build_download_list,
    download_file,
    geojson_stem_to_ndpi_name,
    location_prefix,
    PANGAEA_BASES,
    DEFAULT_GEOJSON_DIR,
    DEFAULT_OUT_DIR,
)

## 1. Configuration

Override `GEOJSON_DIR` or `OUT_DIR` here if your paths differ from the defaults.

In [4]:
GEOJSON_DIR = DEFAULT_GEOJSON_DIR   # data/dataset_28_04
OUT_DIR     = DEFAULT_OUT_DIR        # data/ndpi

print(f"GeoJSON source : {GEOJSON_DIR.resolve()}")
print(f"NDPI output    : {OUT_DIR.resolve()}")
print(f"PANGAEA bases  :")
for loc, url in PANGAEA_BASES.items():
    print(f"  {loc} → {url}")

GeoJSON source : /Users/serenasritharan/Projects/coral-microscopy/data/dataset_28_04
NDPI output    : /Users/serenasritharan/Projects/coral-microscopy/data/ndpi
PANGAEA bases  :
  CHN → https://download.pangaea.de/dataset/984641/files/
  LHP → https://download.pangaea.de/dataset/984640/files/


## 2. Dataset status

Check which NDPI files are already present and which still need downloading.

In [6]:
import pandas as pd

geojson_files = sorted(GEOJSON_DIR.glob("*.geojson"))
print(f"Found {len(geojson_files)} GeoJSON files in {GEOJSON_DIR.name}/\n")

rows = []
for gj in geojson_files:
    stem = gj.stem
    ndpi_name = geojson_stem_to_ndpi_name(stem)
    dest = OUT_DIR / ndpi_name
    loc  = location_prefix(stem)
    url  = PANGAEA_BASES.get(loc, "unknown") + ndpi_name

    size_mb = dest.stat().st_size / 1_048_576 if dest.exists() else None
    rows.append({
        "file"     : ndpi_name,
        "location" : loc,
        "present"  : dest.exists(),
        "size_MB"  : round(size_mb, 1) if size_mb else "-",
        "url"      : url,
    })

status_df = pd.DataFrame(rows)

present = status_df["present"].sum()
missing = (~status_df["present"]).sum()
print(f"  Present : {present}/{len(status_df)}")
print(f"  Missing : {missing}/{len(status_df)}")
print()

# # Highlight missing rows
# def _highlight(row):
#     colour = "background-color: #ffeeba" if not row["present"] else ""
#     return [colour] * len(row)

# status_df.style.apply(_highlight, axis=1)

Found 26 GeoJSON files in dataset_28_04/

  Present : 0/26
  Missing : 26/26



## 3. Build download list

`build_download_list` (from `scripts/download_ndpi.py`) returns only the files that
are not yet present on disk.

In [ ]:
todo = build_download_list(GEOJSON_DIR, OUT_DIR)

if not todo:
    print("All NDPI files already present — nothing to download.")
else:
    print(f"{len(todo)} file(s) queued for download:")
    for url, dest in todo:
        print(f"  {dest.name}")

## 3a. Standalone single-file download

Use this cell when you want one known NDPI without building or filtering the full directory download list. The example below downloads `CHN_AU_10_19-21.ndpi` directly into `data/ndpi/`.

In [1]:
single_stem = "CHN_AU_10_19-21"
single_ndpi_name = geojson_stem_to_ndpi_name(single_stem)
single_loc = location_prefix(single_stem)

if single_loc not in PANGAEA_BASES:
    raise ValueError(f"Unknown location prefix for {single_stem!r}")

single_url = PANGAEA_BASES[single_loc] + single_ndpi_name
single_dest = OUT_DIR / single_ndpi_name

print(f"Target file : {single_dest}")
print(f"Source URL  : {single_url}")

if single_dest.exists() and single_dest.stat().st_size > 1_048_576:
    size_mb = single_dest.stat().st_size / 1_048_576
    print(f"Already present ({size_mb:.1f} MB); skipping download.")
else:
    if single_dest.exists():
        size_mb = single_dest.stat().st_size / 1_048_576
        print(f"Existing file is only {size_mb:.3f} MB; re-downloading.")
    ok = download_file(single_url, single_dest, verbose=True)
    if not ok:
        raise RuntimeError(
            f"Download failed for {single_ndpi_name}. "
            "Re-run the import cell above to load the updated downloader, then try again."
        )


NameError: name 'geojson_stem_to_ndpi_name' is not defined

## 4. Selective download

To download a specific subset, filter `todo` by filename or location prefix before
running the cell below.  Examples:

```python
# Only CHN files
todo_filtered = [(url, dest) for url, dest in todo if "CHN" in dest.name]

# Only a single file
todo_filtered = [(url, dest) for url, dest in todo if dest.name == "CHN_AU_8_7-9.ndpi"]
```

Leave `todo_filtered = todo` to download everything that is missing.

In [ ]:
# ---------- edit this filter as needed ----------
todo_filtered = todo   # download all missing files
# ------------------------------------------------

print(f"Files selected for download: {len(todo_filtered)}")
for url, dest in todo_filtered:
    print(f"  {dest.name:40s}  ({url})")

## 5. Execute downloads

Calls `download_file` from `scripts/download_ndpi.py` for each queued file.
Progress is printed per file. Re-running this cell is safe — files already on disk
will not appear in `todo_filtered`.

In [ ]:
failed = []

for i, (url, dest) in enumerate(todo_filtered, 1):
    print(f"[{i}/{len(todo_filtered)}] {dest.name}")
    ok = download_file(url, dest, verbose=True)
    if not ok:
        failed.append(dest.name)
    print()

print("-" * 50)
if failed:
    print(f"Failed ({len(failed)}/{len(todo_filtered)}):")
    for name in failed:
        print(f"  {name}")
else:
    if todo_filtered:
        print(f"All {len(todo_filtered)} file(s) downloaded successfully.")
    else:
        print("Nothing to download.")

## 6. Post-download status

Re-check which files are now present and report total disk usage.

In [ ]:
ndpi_files = sorted(OUT_DIR.glob("*.ndpi")) if OUT_DIR.exists() else []
total_gb   = sum(f.stat().st_size for f in ndpi_files) / 1_073_741_824

print(f"NDPI files present in {OUT_DIR.name}/: {len(ndpi_files)}")
print(f"Total disk usage: {total_gb:.2f} GB")
print()

for f in ndpi_files:
    size_mb = f.stat().st_size / 1_048_576
    # Read basic metadata via OpenSlide if available
    try:
        import openslide
        slide = openslide.OpenSlide(str(f))
        w, h  = slide.dimensions
        mpp_x = slide.properties.get(openslide.PROPERTY_NAME_MPP_X, "?")
        mpp_y = slide.properties.get(openslide.PROPERTY_NAME_MPP_Y, "?")
        levels = len(slide.level_dimensions)
        slide.close()
        meta = f"{w}×{h} px  mpp=({mpp_x},{mpp_y})  {levels} levels"
    except Exception:
        meta = "(openslide unavailable)"

    print(f"  {f.name:40s}  {size_mb:8.1f} MB   {meta}")